# 心思维 AI 扒谱助手｜唯一 Colab 工作入口

**正式分支：** `feature/vocalparse-event-musicxml`  
**当前工作：** 从 Google Drive 读取已经生成的 VocalParse MusicXML，进行调号/移调后续处理，并生成简谱与五线谱。  
**运行原则：** 不重新扒谱、不重新下载音频模型、不覆盖历史文件；办公室和家中统一使用本 Notebook 与 GitHub 正式分支。

本 Notebook 只保留日常工作必需步骤：连接 Drive → 获取项目代码 → 准备谱面排版环境 → 读取 MusicXML 并输出谱面。


## 01｜连接 Google Drive

运行下面单元格并完成授权。

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
MY_DRIVE = Path('/content/drive/MyDrive')
if not MY_DRIVE.is_dir():
    raise RuntimeError('Google Drive 未成功挂载。')
print('Google Drive 已连接：', MY_DRIVE)


## 02｜连接 GitHub 正式工作分支

从 Colab Secrets 读取 `GH_TOKEN`，安全获取私有仓库代码。不会把 Token 写入 Notebook、命令行参数或 Git 持久配置；已有仓库若分支不符则停止，不自动切换。

In [ ]:
from pathlib import Path
import base64
import json
import os
import shutil
import subprocess
import sys
import urllib.error
import urllib.request

REPO_DIR = Path("/content/xinsw-music-transcriber")
STAGING_DIR = Path("/content/.xinsw-music-transcriber-clone")
FAILED_DIR = Path("/content/.xinsw-music-transcriber-clone-failed")
REPO_URL = "https://github.com/Trifida/xinsw-music-transcriber.git"
REPO_API = "https://api.github.com/repos/Trifida/xinsw-music-transcriber"
REPO_BRANCH = "feature/vocalparse-event-musicxml"

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(
            f"目标目录已存在但不是 Git 仓库：{REPO_DIR}。"
            "为保护现有文件，Notebook 不会删除或覆盖它。"
        )
    current_branch = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True
    ).strip()
    if current_branch != REPO_BRANCH:
        raise RuntimeError(
            f"现有仓库分支为 {current_branch!r}，预期为 {REPO_BRANCH!r}。"
            "为保护现有工作，Notebook 不会自动切换分支。"
        )
    print("发现已有项目仓库，保留原样：", REPO_DIR)
else:
    from google.colab import userdata
    try:
        gh_token = userdata.get("GH_TOKEN")
    except Exception as exc:
        raise RuntimeError(
            "尚未取得 Colab Secret：GH_TOKEN。请在 Colab 左侧 Secrets 中添加该名称，"
            "并允许本主 Notebook 访问。不要把 Token 粘贴到代码或聊天中。"
        ) from exc
    if not gh_token:
        raise RuntimeError("Colab Secret GH_TOKEN 为空，请检查 Secret 设置。")

    # 先用 GitHub API 验证 Token 是否确实能读取这个私有仓库，不显示 Token。
    api_request = urllib.request.Request(
        REPO_API,
        headers={"Authorization": f"Bearer {gh_token}", "Accept": "application/vnd.github+json"}
    )
    try:
        with urllib.request.urlopen(api_request, timeout=30) as response:
            repo_info = json.loads(response.read().decode("utf-8"))
        print("GitHub 私有仓库读取验证通过：", repo_info.get("full_name", "仓库已识别"))
    except urllib.error.HTTPError as exc:
        del gh_token
        if exc.code == 404:
            raise RuntimeError(
                "GitHub 返回 404：当前 Token 看不到该私有仓库。请检查创建 Token 时是否选中了 "
                "Trifida/xinsw-music-transcriber，以及 Resource owner 是否正确。"
            ) from None
        if exc.code == 403:
            raise RuntimeError(
                "GitHub 返回 403：Token 可能尚未获准访问该仓库，或组织要求额外批准。"
                "请检查 Token 的仓库访问范围和组织授权。"
            ) from None
        raise RuntimeError(f"GitHub 仓库读取验证失败，HTTP {exc.code}。") from None
    except Exception as exc:
        del gh_token
        raise RuntimeError(f"GitHub 仓库读取验证未完成：{type(exc).__name__}: {exc}") from None

    # 上一步已验证 PAT 可读仓库。PAT 使用 Git HTTPS Basic：GitHub 登录名:Token。
    auth_value = "AUTHORIZATION: basic " + base64.b64encode(
        ("Trifida:" + gh_token).encode("utf-8")
    ).decode("ascii")
    del gh_token

    # 上一次失败的 clone 可能留下临时目录。先改名保留，不删除任何文件。
    if STAGING_DIR.exists():
        if FAILED_DIR.exists():
            raise RuntimeError(
                f"发现旧临时目录 {STAGING_DIR}，且备份目录 {FAILED_DIR} 也已存在。"
                "为保护文件，停止操作，请先检查这两个目录。"
            )
        STAGING_DIR.rename(FAILED_DIR)
        print("已将上次失败的临时下载目录保留为：", FAILED_DIR)

    git_env = os.environ.copy()
    git_env["GIT_CONFIG_COUNT"] = "1"
    git_env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
    git_env["GIT_CONFIG_VALUE_0"] = auth_value
    del auth_value
    result = subprocess.run(
        ["git", "clone", "--single-branch", "--branch", REPO_BRANCH, REPO_URL, str(STAGING_DIR)],
        env=git_env, text=True, capture_output=True
    )
    del git_env
    if result.returncode != 0:
        detail = result.stderr.strip() or result.stdout.strip() or "未提供详细错误信息"
        raise RuntimeError("GitHub 私有仓库克隆失败：\n" + detail)
    if not (STAGING_DIR / ".git").is_dir():
        raise RuntimeError("下载结束但临时目录中没有 Git 元数据；保留现场，不自动清理。")
    shutil.move(str(STAGING_DIR), str(REPO_DIR))
    print("项目代码获取完成：", REPO_DIR)

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True
).strip()
print("项目分支：", REPO_BRANCH)
print("当前提交：", commit)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

## 03｜准备谱面排版环境

只在缺少组件时安装 Verovio、jianpu-ly 和 LilyPond；不安装、不下载任何音频识别模型。

In [ ]:
import importlib.util
import os
import shutil
import subprocess
import sys

if 'REPO_DIR' not in globals() or not (REPO_DIR / 'musicmaster' / 'core' / 'render.py').is_file():
    raise RuntimeError('请先运行 02｜连接 GitHub 正式工作分支。')

missing_python = []
if importlib.util.find_spec('verovio') is None:
    missing_python.append('verovio>=4.0')
if importlib.util.find_spec('jianpu_ly') is None:
    missing_python.append('jianpu-ly>=1.8')
if missing_python:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing_python], check=True)

if shutil.which('lilypond') is None:
    apt_get = shutil.which('apt-get')
    if apt_get is None:
        raise RuntimeError('当前不是支持 apt-get 的 Colab Linux 运行环境。')
    if hasattr(os, 'geteuid') and os.geteuid() != 0:
        raise RuntimeError('当前运行时没有安装 LilyPond 所需的系统权限。')
    subprocess.run([apt_get, 'update', '-qq'], check=True)
    subprocess.run([apt_get, 'install', '-y', 'lilypond'], check=True)

importlib.invalidate_caches()
backend_status = {
    'verovio': importlib.util.find_spec('verovio') is not None,
    'jianpu_ly': importlib.util.find_spec('jianpu_ly') is not None,
    'lilypond': shutil.which('lilypond') is not None,
}
print('谱面排版环境：', backend_status)
if not all(backend_status.values()):
    raise RuntimeError('谱面排版环境未全部就绪。')


## 04｜读取已有 MusicXML，生成简谱与五线谱

默认使用 2026-11-23 已保存的 `vocalparse.musicxml`。如后续处理了新歌曲，只需修改下一单元格中的 `MUSICXML_PATH`；每次输出到独立新目录，不覆盖原文件。

In [ ]:
from pathlib import Path
import json
import sys
from datetime import datetime, timezone

if 'MY_DRIVE' not in globals() or not Path(MY_DRIVE).is_dir():
    raise RuntimeError('请先运行 01｜连接 Google Drive。')
if 'REPO_DIR' not in globals() or not (REPO_DIR / 'musicmaster' / 'core' / 'render.py').is_file():
    raise RuntimeError('请先运行 02｜连接 GitHub 正式工作分支。')
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# 当前已验证的 MusicXML；处理新歌曲时只需修改这一行。
MUSICXML_PATH = MY_DRIVE / 'XinsW_Lead_Transcription' / 'VocalParse_Results' / 'restore_20261123T150049_359290Z' / 'vocalparse.musicxml'
MUSICXML_PATH = Path(MUSICXML_PATH)
if not MUSICXML_PATH.is_file():
    raise FileNotFoundError(f'找不到输入 MusicXML：{MUSICXML_PATH}')

from musicmaster.core.render import render, is_jianpu_source_available, lilypond_executable

render_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
render_dir = MUSICXML_PATH.parent / f'score_render_{render_id}'
render_dir.mkdir(parents=True, exist_ok=False)

artifacts = render(
    MUSICXML_PATH,
    render_dir,
    formats=('staff', 'jianpu'),
    jianpu_format='svg',
    require_all=True,
)

artifact_report = {}
for name, output_path in artifacts.items():
    output_path = Path(output_path)
    if not output_path.is_file() or output_path.stat().st_size == 0:
        raise RuntimeError(f'输出文件无效：{name} -> {output_path}')
    artifact_report[name] = {
        'path': str(output_path),
        'size_bytes': output_path.stat().st_size,
    }

report = {
    'status': 'passed',
    'input_musicxml': str(MUSICXML_PATH),
    'output_directory': str(render_dir),
    'jianpu_source_available': is_jianpu_source_available(),
    'lilypond_available': bool(lilypond_executable()),
    'outputs': artifact_report,
}
report_path = render_dir / 'render_report.json'
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print('简谱 / 五线谱生成完成。')
print(json.dumps(report, ensure_ascii=False, indent=2))
